## Download CIMIS data

In [21]:
import requests
import pandas as pd
import time
from pathlib import Path

APP_KEY = "45312222-b15a-4a94-b5ed-bc5ee049c555"

OUTDIR = Path("cimis_hourly")

# Create directory only if it doesn't already exist
OUTDIR.mkdir(parents=True, exist_ok=True)

DATA_ITEMS = [
    "hly-air-tmp",
    "hly-rel-hum",
    "hly-wind-spd",
    "hly-sol-rad"
]

# -----------------------------------------------------
# GET STATIONS
# -----------------------------------------------------
session = requests.Session()

station_url = (
    f"https://et.water.ca.gov/api/station?"
    f"appKey={APP_KEY}"
)

stations = session.get(station_url).json()["Stations"]

meta_df = pd.DataFrame([{
    "station_id": s["StationNbr"],
    "station_name": s["Name"],
    "latitude": s["HmsLatitude"],
    "longitude": s["HmsLongitude"],
    "elevation_ft": s["Elevation"],
    "is_active": s["IsActive"]
}
for s in stations
if s["IsActive"] == "True"
])

# -----------------------------------------------------
# LOOP YEARS
# -----------------------------------------------------

for year in range(2000, 2026):  # 2000 to 2026

    year_dir = OUTDIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nYEAR {year}")

    for _, station in meta_df.iterrows():

        sid = station["station_id"]

        outfile = year_dir / f"station_{sid}_{year}.csv"
        
        # Skip if this station has already been downloaded in this year
        if outfile.exists():
            print(f"Skipping {outfile.name}")
            continue

        print(f"Downloading station {sid}")
        
        yearly_data = []

        # -------------------------------------------------
        # LOOP MONTHS
        # -------------------------------------------------

        import calendar
        
        for month in range(1, 13):
        
            last_day = calendar.monthrange(year, month)[1]
        
            start_date = f"{year}-{month:02d}-01"
            end_date   = f"{year}-{month:02d}-{last_day}"
        
            data_url = (
                "https://et.water.ca.gov/api/data"
                f"?appKey={APP_KEY}"
                f"&targets={sid}"
                f"&startDate={start_date}"
                f"&endDate={end_date}"
                f"&dataItems={','.join(DATA_ITEMS)}"
                "&unitOfMeasure=M"
            )
                    
            try:
            
                max_retries = 3
                for attempt in range(max_retries):
                    try:
                        r = session.get(data_url, timeout=60)
                        break
                    except (requests.exceptions.ReadTimeout,
                            requests.exceptions.ConnectionError
                            ):
                        print(
                            f"    retry {attempt+1}/{max_retries}"
                        )
                        time.sleep(5)
                        
                else:
                    print("    FAILED after retries")
                    continue
            
                print(f"  {year}-{month:02d} -> {r.status_code}")
            
                if r.status_code != 200:
                    print(r.text[:300])
                    continue
            
                j = r.json()
            
                providers = j.get("Data", {}).get("Providers", [])
            
                if len(providers) == 0:
                    print("    no providers")
                    continue
            
                records = providers[0].get("Records", [])
            
                if len(records) == 0:
                    print("    no records")
                    continue
            
                df_month = pd.DataFrame(records)
            
                yearly_data.append(df_month)
            
            except Exception as e:
            
                print(f"    FAILED: {e}")        
            
            time.sleep(1)

        # -------------------------------------------------
        # SAVE YEAR
        # -------------------------------------------------

        if len(yearly_data) == 0:

            print("  no yearly data")
            continue

        df = pd.concat(yearly_data, ignore_index=True)

        df["station_id"] = sid
        df["station_name"] = station["station_name"]
        df["latitude"] = station["latitude"]
        df["longitude"] = station["longitude"]
        df["elevation_ft"] = station["elevation_ft"]

        df.to_csv(outfile, index=False)

        print(f"  SAVED {outfile}")

print("DONE")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)